# Module 1 - Verify the Mac MPS Environment

**Audience:** engineers reproducing this project on an Apple Silicon Mac.

**Prerequisites:** Python 3.12, the project virtual environment, and the VS Code Jupyter extension.

**Learning goals:** verify the selected Python kernel, detect the MPS backend, exercise the chosen device, and capture runtime metadata for later experiment manifests.


## Outline

1. Locate the repository and import the project package.
2. Inspect Python, PyTorch and Apple MPS availability.
3. Exercise the selected device with a small matrix operation.
4. Verify repeatable seeding.
5. Review pitfalls before model training.


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

current = Path.cwd().resolve()
candidates = [current, *current.parents]
PROJECT_ROOT = next(path for path in candidates if (path / 'pyproject.toml').exists())
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

PROJECT_ROOT


## Step 1 - Inspect the runtime

The project uses `mps` for the Apple GPU and falls back to CPU in environments such as GitHub Actions. An explicit request for unavailable MPS fails fast so a long training run cannot silently execute on CPU.


In [ ]:
import torch

from governed_banking.device import seed_everything, select_device

device, runtime = select_device('auto')
print(json.dumps(runtime.to_dict(), indent=2))
print(f'Selected device: {device}')


### Interpretation

On the MacBook Pro, `mps_built` and `mps_available` should both be `true`, and `selected` should be `mps`. A restricted automation environment may select CPU even when the physical Mac supports MPS; the VS Code notebook result is authoritative for local training.


## Step 2 - Exercise the selected device

This is intentionally small. It verifies tensor placement without creating unnecessary memory pressure.


In [ ]:
seed_everything(42)
matrix = torch.randn(512, 512, device=device)
result = matrix @ matrix.T
if device.type == 'mps':
    torch.mps.synchronize()

assert result.device.type == device.type
summary = {
    'shape': tuple(result.shape),
    'device': str(result.device),
    'finite': bool(result.isfinite().all()),
}
print(summary)


## Step 3 - Verify deterministic setup

Identical seeds should reproduce this small tensor within one runtime. Exact equality across operating systems, devices or library versions is not guaranteed, which is why every later experiment will record its environment.


In [ ]:
seed_everything(17)
first = torch.rand(8, device=device)
seed_everything(17)
second = torch.rand(8, device=device)

assert torch.equal(first.cpu(), second.cpu())
print('Seed check passed.')


## Exercise

Change the preferred device below to `mps` and run it directly in VS Code. Explain why an explicit request should fail instead of falling back when MPS is unavailable.


In [ ]:
preferred = 'auto'  # Change to 'mps' for the exercise.
exercise_device, exercise_runtime = select_device(preferred)
exercise_runtime.to_dict()


## Pitfalls and next step

- Do not select a system Python kernel when the project `.venv` is available.
- Do not use CUDA-only settings or install `bitsandbytes` for this first MPS implementation.
- Do not disable MPS memory safeguards to force a larger batch into memory.
- Do not begin model training until the data split and claims protocol are committed.

After this notebook passes locally, Module 2 will implement an immutable BANKING77 loader and leakage tests.
